In [ ]:
# Project : Factory Optimization using Nassau Candy Distributor Dataset
#
# Objectives:
# 1. Predict Sales using Random Forest Regressor
# 2. Remove Data Leakage
# 3. Use Professional ML Pipeline
# 4. Apply One-Hot Encoding
# 5. Perform Hyperparameter Tuning
# 6. Evaluate the Model
# 7. Perform Cross Validation
# 8. Display Feature Importance
# 9. Save the Trained Pipeline
# =============================================================================

# =============================================================================
# STEP 1 - IMPORT REQUIRED LIBRARIES
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Model Selection
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    cross_val_score
)

# Pipeline
from sklearn.pipeline import Pipeline

# Column Transformer
from sklearn.compose import ColumnTransformer

# Missing Value Handling
from sklearn.impute import SimpleImputer

# Encoding
from sklearn.preprocessing import OneHotEncoder

# Machine Learning Model
from sklearn.ensemble import RandomForestRegressor

# Evaluation Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =============================================================================
# STEP 2 - LOAD FEATURE ENGINEERED DATASET
# =============================================================================

print("=" * 70)
print("Loading Dataset...")
print("=" * 70)

file_path = r"C:/Users/aryan/Downloads/feature_engineered_nassau_candy.csv"

df = pd.read_csv(file_path)

print("\nDataset Loaded Successfully")
print("Shape :", df.shape)

# Display first five records
print(df.head())

# =============================================================================
# STEP 3 - REMOVE DATA LEAKAGE
# =============================================================================
#
# Data leakage occurs when we use information that would not be available
# at prediction time.
#
# These columns are derived from Sales or are too closely related.
# They must NOT be used while predicting Sales.
#
# =============================================================================

leakage_columns = [

    "Sales",

    "Gross Profit",

    "Cost",

    "Revenue Check",

    "Sales Per Unit",

    "Cost Per Unit",

    "Profit Per Unit",

    "Profit Margin (%)",

    "Sales Category",

    "Profit Category",

    "High Demand"

]

# =============================================================================
# STEP 4 - DEFINE TARGET VARIABLE
# =============================================================================

# Target variable

y = df["Sales"]

# =============================================================================
# STEP 5 - SELECT FEATURES
# =============================================================================

# Remove unnecessary columns

X = df.drop(

    columns=leakage_columns +

    [

        "Order ID",

        "Product ID"

    ]

)

print("\nFeatures Selected")

print(X.columns)

# =============================================================================
# STEP 6 - IDENTIFY CATEGORICAL AND NUMERICAL COLUMNS
# =============================================================================

categorical_features = X.select_dtypes(

    include="object"

).columns

numeric_features = X.select_dtypes(

    exclude="object"

).columns

print("\nCategorical Features")

print(categorical_features)

print("\nNumerical Features")

print(numeric_features)

# =============================================================================
# STEP 7 - CREATE PREPROCESSING PIPELINE
# =============================================================================

# Numerical preprocessing
#
# Fill missing numerical values using Median

numeric_transformer = Pipeline([

    (

        "imputer",

        SimpleImputer(strategy="median")

    )

])

# Categorical preprocessing
#
# Fill missing values
# Convert text into One-Hot Encoding

categorical_transformer = Pipeline([

    (

        "imputer",

        SimpleImputer(

            strategy="most_frequent"

        )

    ),

    (

        "encoder",

        OneHotEncoder(

            handle_unknown="ignore"

        )

    )

])

# =============================================================================
# STEP 8 - COMBINE BOTH PREPROCESSORS
# =============================================================================

preprocessor = ColumnTransformer(

    transformers=[

        (

            "num",

            numeric_transformer,

            numeric_features

        ),

        (

            "cat",

            categorical_transformer,

            categorical_features

        )

    ]

)

# =============================================================================
# STEP 9 - CREATE MACHINE LEARNING PIPELINE
# =============================================================================

pipeline = Pipeline([

    (

        "preprocessor",

        preprocessor

    ),

    (

        "model",

        RandomForestRegressor(

            random_state=42

        )

    )

])

# =============================================================================
# STEP 10 - TRAIN TEST SPLIT
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42

)

print("\nTraining Samples :", len(X_train))

print("Testing Samples :", len(X_test))

# =============================================================================
# STEP 11 - HYPERPARAMETER TUNING
# =============================================================================

parameter_grid = {

    "model__n_estimators":

        [100, 200, 300],

    "model__max_depth":

        [10, 20, 30, None],

    "model__min_samples_split":

        [2, 5, 10],

    "model__min_samples_leaf":

        [1, 2, 4]

}

print("\nSearching Best Parameters...")

search = RandomizedSearchCV(

    estimator=pipeline,

    param_distributions=parameter_grid,

    n_iter=10,

    cv=5,

    scoring="r2",

    random_state=42,

    n_jobs=-1

)

search.fit(

    X_train,

    y_train

)

# Best model

best_model = search.best_estimator_

print("\nBest Parameters")

print(search.best_params_)

# =============================================================================
# STEP 12 - PREDICT SALES
# =============================================================================

sales_prediction = best_model.predict(

    X_test

)

# =============================================================================
# STEP 13 - MODEL EVALUATION
# =============================================================================

print("\n")

print("=" * 70)

print("MODEL PERFORMANCE")

print("=" * 70)

print("\nMean Absolute Error")

print(

    mean_absolute_error(

        y_test,

        sales_prediction

    )

)

print("\nRoot Mean Squared Error")

print(

    np.sqrt(

        mean_squared_error(

            y_test,

            sales_prediction

        )

    )

)

print("\nR2 Score")

print(

    r2_score(

        y_test,

        sales_prediction

    )

)

# =============================================================================
# STEP 14 - CROSS VALIDATION
# =============================================================================

scores = cross_val_score(

    best_model,

    X,

    y,

    cv=5,

    scoring="r2"

)

print("\nCross Validation Scores")

print(scores)

print("\nAverage R2 Score")

print(scores.mean())

# =============================================================================
# STEP 15 - FEATURE IMPORTANCE
# =============================================================================

feature_names = best_model.named_steps[

    "preprocessor"

].get_feature_names_out()

importance = best_model.named_steps[

    "model"

].feature_importances_

importance_df = pd.DataFrame({

    "Feature": feature_names,

    "Importance": importance

})

importance_df = importance_df.sort_values(

    by="Importance",

    ascending=False

)

print("\nTop 20 Important Features")

print(importance_df.head(20))

# =============================================================================
# STEP 16 - FEATURE IMPORTANCE GRAPH
# =============================================================================

top20 = importance_df.head(20)

plt.figure(figsize=(10,8))

plt.barh(

    top20["Feature"],

    top20["Importance"]

)

plt.gca().invert_yaxis()

plt.xlabel("Importance")

plt.ylabel("Feature")

plt.title("Top 20 Features Affecting Sales")

plt.tight_layout()

plt.show()

# =============================================================================
# STEP 17 - SAVE TRAINED PIPELINE
# =============================================================================

model_path = r"C:/Users/aryan/Downloads/factory_sales_pipeline.pkl"

joblib.dump(

    best_model,

    model_path

)

print("\nPipeline Saved Successfully")

print(model_path)

# =============================================================================
# STEP 18 - LOAD SAVED PIPELINE
# =============================================================================

loaded_model = joblib.load(model_path)

print("\nSaved Pipeline Loaded Successfully")

# =============================================================================
# STEP 19 - SAMPLE PREDICTION
# =============================================================================

sample_prediction = loaded_model.predict(

    X.head()

)

print("\nSample Predictions")

print(sample_prediction)

# =============================================================================
# MODULE COMPLETED
# =============================================================================

print("\n" + "=" * 70)
print("MODULE 5.1 COMPLETED SUCCESSFULLY")
print("=" * 70)

Loading Dataset...

Dataset Loaded Successfully
Shape : (10194, 36)
   Row ID                      Order ID  Order Date   Ship Date  \
0       1  US-2021-103800-CHO-MIL-31000  2024-01-03  2026-06-30   
1       2  US-2021-112326-CHO-TRI-54000  2024-01-04  2026-07-01   
2       3  US-2021-112326-CHO-NUT-13000  2024-01-04  2026-07-01   
3       4  US-2021-112326-CHO-SCR-58000  2024-01-04  2026-07-01   
4       5  US-2021-141817-CHO-TRI-54000  2024-01-05  2026-07-05   

        Ship Mode  Customer ID Country/Region          City State/Province  \
0  Standard Class       103800  United States       Houston          Texas   
1  Standard Class       112326  United States    Naperville       Illinois   
2  Standard Class       112326  United States    Naperville       Illinois   
3  Standard Class       112326  United States    Naperville       Illinois   
4  Standard Class       141817  United States  Philadelphia   Pennsylvania   

  Postal Code  ... Sales Per Unit Cost Per Unit Profit Per U

C:\Users\aryan\AppData\Local\Temp\ipykernel_18808\2124741550.py:145: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(
